# OSM Urban Feature Extraction

This notebook extracts urban morphology features from OpenStreetMap for each of the 30 European cities in the study. These features serve as **cross-sectional predictors** of light pollution in the regression analysis (Notebook 08).

Features are extracted within a **25 km buffer** around each city center, matching the VIIRS spatial aggregation.

**What this notebook does:**
- Extracts road network metrics (total length, motorway length, density)
- Counts and measures building footprints
- Measures land use areas (residential, commercial, industrial, retail)
- Measures green/dark areas (parks, forests, water, nature reserves)
- Counts street lamps
- Saves a tidy cross-sectional CSV to `../data/processed/osm_city_features.csv`

**Note:** OSM provides a current snapshot, not historical data. These features are treated as time-invariant urban characteristics.

In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import numpy as np
import matplotlib.pyplot as plt
import requests as req
import time
import os
import json
import warnings
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=FutureWarning)

# Configuration
BUFFER_DIST = 25_000       # meters, matching VIIRS extraction
BUILDING_INNER_DIST = 10_000  # meters, inner buffer for building geometry (faster)
CRS_PROJ = "EPSG:3035"    # ETRS89-LAEA Europe (equal-area)
CRS_WGS84 = "EPSG:4326"
SLEEP_SECONDS = 4          # rate limit between Overpass API calls
INTERIM_DIR = "../data/interim/osm"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
os.makedirs(INTERIM_DIR, exist_ok=True)

# Increase osmnx Overpass timeout
ox.settings.overpass_rate_limit = False
ox.settings.timeout = 300

# Buffer area in km2 (circle with r=25km)
BUFFER_AREA_APPROX_KM2 = np.pi * (BUFFER_DIST / 1000) ** 2

print(f"osmnx {ox.__version__}, geopandas {gpd.__version__}")
print(f"Buffer distance: {BUFFER_DIST:,} m (buildings inner: {BUILDING_INNER_DIST:,} m)")
print(f"Approximate buffer area: {BUFFER_AREA_APPROX_KM2:.0f} km²")

In [ ]:
cities = pd.read_csv("../data/processed/cities.csv")
print(f"Loaded {len(cities)} cities")
cities

## Helper: caching and rate limiting

Raw GeoDataFrames from Overpass are cached to `data/interim/osm/` as GeoJSON so that re-running the notebook does not re-query the API.

In [ ]:
def _cache_path(city: str, feature: str) -> str:
    """Return the cache file path for a city/feature pair."""
    safe_name = city.lower().replace(" ", "_")
    return os.path.join(INTERIM_DIR, f"{safe_name}_{feature}.geojson")


def _save_cache(gdf: gpd.GeoDataFrame, city: str, feature: str) -> None:
    path = _cache_path(city, feature)
    gdf.to_file(path, driver="GeoJSON")


def _load_cache(city: str, feature: str) -> gpd.GeoDataFrame | None:
    path = _cache_path(city, feature)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return gpd.read_file(path)
    return None


def _sleep():
    time.sleep(SLEEP_SECONDS)

## Road network extraction

Uses `osmnx.graph_from_point()` to download the full road network within the 25 km buffer, then computes total road length and motorway/trunk length from the edges GeoDataFrame.

In [ ]:
def extract_roads(lat: float, lon: float, city: str) -> dict:
    """
    Extract road network metrics within BUFFER_DIST of (lat, lon).
    Returns dict with road_length_total_km and motorway_length_km.
    """
    cached = _load_cache(city, "roads")
    if cached is not None:
        edges = cached
    else:
        try:
            G = ox.graph_from_point((lat, lon), dist=BUFFER_DIST, network_type="drive")
            edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
            _save_cache(edges, city, "roads")
            _sleep()
        except Exception as e:
            print(f"  WARNING: road extraction failed for {city}: {e}")
            return {"road_length_total_km": np.nan, "motorway_length_km": np.nan}

    # Project to equal-area CRS for length calculation
    edges_proj = edges.to_crs(CRS_PROJ)
    edges_proj["length_m"] = edges_proj.geometry.length

    total_km = edges_proj["length_m"].sum() / 1000

    # Motorway + trunk roads
    if "highway" in edges_proj.columns:
        motorway_mask = edges_proj["highway"].apply(
            lambda x: any(v in ([x] if isinstance(x, str) else (x if isinstance(x, list) else []))
                         for v in ["motorway", "trunk", "motorway_link", "trunk_link"])
        )
        motorway_km = edges_proj.loc[motorway_mask, "length_m"].sum() / 1000
    else:
        motorway_km = 0.0

    return {
        "road_length_total_km": round(total_km, 2),
        "motorway_length_km": round(motorway_km, 2),
    }

## Building footprint extraction (two-tier approach)

Downloading full building polygon geometry at 25 km can take 30-60 min per city (500k+ polygons for dense cities). To keep runtime practical while preserving `building_area_km2`:

1. **If a full 25 km cache exists** (e.g., from a prior run): load it and compute exact count + area.
2. **Otherwise, two-tier query:**
   - **Count** at 25 km via a fast Overpass `out count` query (~30 seconds)
   - **Full geometry** at 10 km inner buffer to compute building area in the dense urban core (~5-8 min)

The 10 km core captures the vast majority of building footprint area because the outer 10-25 km ring is typically low-density suburban/rural.

In [ ]:
def _overpass_count(query: str, max_retries: int = 3) -> int | None:
    """Send a count query to Overpass with retry logic for 429/504."""
    for attempt in range(max_retries):
        try:
            r = req.post(OVERPASS_URL, data={"data": query}, timeout=180)
            if r.status_code == 200:
                return int(r.json()["elements"][0]["tags"]["total"])
            elif r.status_code in (429, 504):
                wait = 30 * (attempt + 1)
                print(f"    Overpass {r.status_code}, retrying in {wait}s ({attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                print(f"    Overpass HTTP {r.status_code}")
                return None
        except Exception as e:
            print(f"    Overpass error: {e}")
            time.sleep(15)
    return None


def _compute_building_area(gdf: gpd.GeoDataFrame) -> float:
    """Compute total building footprint area in km2 from a GeoDataFrame."""
    polys = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    if len(polys) == 0:
        return 0.0
    polys_proj = polys.to_crs(CRS_PROJ)
    return polys_proj.geometry.area.sum() / 1e6


def extract_buildings(lat: float, lon: float, city: str) -> dict:
    """
    Extract building count and footprint area.
    
    Strategy:
    1. If a full 25km cache exists (from prior run): use it for exact count + area.
    2. Otherwise: count at 25km (fast Overpass query) + geometry at 10km (area).
    """
    # --- Check for full 25km legacy cache (e.g., Madrid, Barcelona) ---
    legacy_cache = _cache_path(city, "buildings")
    if os.path.exists(legacy_cache) and os.path.getsize(legacy_cache) > 0:
        print(f"  [buildings] Loading full 25km cache for {city}")
        gdf = gpd.read_file(legacy_cache)
        polys = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
        return {
            "building_count": len(polys),
            "building_area_km2": round(_compute_building_area(gdf), 4),
        }

    # --- Check for two-tier cache ---
    tier_cache = os.path.join(INTERIM_DIR, f"{city.lower().replace(' ', '_')}_buildings_twotier.json")
    if os.path.exists(tier_cache):
        with open(tier_cache) as f:
            return json.load(f)

    # --- Tier 1: count at 25km (fast) ---
    print(f"  [buildings] Counting at {BUFFER_DIST/1000:.0f}km...")
    count_query = f"""
    [out:json][timeout:180];
    (
      way["building"](around:{BUFFER_DIST},{lat},{lon});
      relation["building"](around:{BUFFER_DIST},{lat},{lon});
    );
    out count;
    """
    count = _overpass_count(count_query)
    if count is None:
        print(f"  WARNING: building count failed for {city}")
        return {"building_count": np.nan, "building_area_km2": np.nan}

    _sleep()

    # --- Tier 2: full geometry at 10km for area ---
    print(f"  [buildings] Downloading geometry at {BUILDING_INNER_DIST/1000:.0f}km for area...")
    inner_cache = _cache_path(city, "buildings_10km")
    if inner_cache and os.path.exists(inner_cache) and os.path.getsize(inner_cache) > 0:
        gdf_inner = gpd.read_file(inner_cache)
    else:
        try:
            gdf_inner = ox.features_from_point(
                (lat, lon), tags={"building": True}, dist=BUILDING_INNER_DIST
            )
            _save_cache(gdf_inner, city, "buildings_10km")
            _sleep()
        except Exception as e:
            print(f"  WARNING: 10km building geometry failed for {city}: {e}")
            gdf_inner = None

    area_km2 = round(_compute_building_area(gdf_inner), 4) if gdf_inner is not None else np.nan

    result = {"building_count": count, "building_area_km2": area_km2}

    # Cache the two-tier result
    with open(tier_cache, "w") as f:
        json.dump(result, f)

    return result

## Land use extraction

Queries land use polygons for residential, commercial, industrial, and retail areas. Overlapping polygons within each category are dissolved before computing area to avoid double-counting.

In [ ]:
LANDUSE_TYPES = ["residential", "commercial", "industrial", "retail"]


def extract_landuse(lat: float, lon: float, city: str) -> dict:
    """
    Extract land use areas within BUFFER_DIST of (lat, lon).
    Returns dict with landuse_{type}_km2 for each type.
    """
    cached = _load_cache(city, "landuse")
    if cached is not None:
        gdf = cached
    else:
        try:
            gdf = ox.features_from_point(
                (lat, lon),
                tags={"landuse": LANDUSE_TYPES},
                dist=BUFFER_DIST,
            )
            _save_cache(gdf, city, "landuse")
            _sleep()
        except Exception as e:
            print(f"  WARNING: landuse extraction failed for {city}: {e}")
            return {f"landuse_{t}_km2": np.nan for t in LANDUSE_TYPES}

    result = {}
    polys = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

    for lu_type in LANDUSE_TYPES:
        subset = polys[polys["landuse"] == lu_type] if "landuse" in polys.columns else polys.iloc[0:0]
        if len(subset) == 0:
            result[f"landuse_{lu_type}_km2"] = 0.0
        else:
            subset_proj = subset.to_crs(CRS_PROJ)
            dissolved = subset_proj.union_all()  # dissolve overlaps
            area_km2 = dissolved.area / 1e6
            result[f"landuse_{lu_type}_km2"] = round(area_km2, 4)

    return result

## Green / dark area extraction

Extracts areas that absorb rather than produce light: parks, forests, water bodies, and nature reserves. These are expected to have **negative** coefficients in the light pollution regression.

In [ ]:
def extract_green_areas(lat: float, lon: float, city: str) -> dict:
    """
    Extract green/dark area metrics within BUFFER_DIST of (lat, lon).
    Returns dict with park_area_km2, forest_area_km2, water_area_km2,
    nature_reserve_area_km2.
    """
    cached = _load_cache(city, "green")
    if cached is not None:
        gdf = cached
    else:
        try:
            gdf = ox.features_from_point(
                (lat, lon),
                tags={
                    "leisure": ["park", "garden", "nature_reserve"],
                    "landuse": ["forest"],
                    "natural": ["wood", "water"],
                },
                dist=BUFFER_DIST,
            )
            _save_cache(gdf, city, "green")
            _sleep()
        except Exception as e:
            print(f"  WARNING: green area extraction failed for {city}: {e}")
            return {
                "park_area_km2": np.nan,
                "forest_area_km2": np.nan,
                "water_area_km2": np.nan,
                "nature_reserve_area_km2": np.nan,
            }

    polys = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

    def _area(mask):
        subset = polys[mask]
        if len(subset) == 0:
            return 0.0
        proj = subset.to_crs(CRS_PROJ)
        dissolved = proj.union_all()
        return round(dissolved.area / 1e6, 4)

    # Parks & gardens
    park_mask = polys.get("leisure", pd.Series(dtype=str)).isin(["park", "garden"])
    # Forests
    forest_mask = (
        polys.get("landuse", pd.Series(dtype=str)).eq("forest")
        | polys.get("natural", pd.Series(dtype=str)).eq("wood")
    )
    # Water
    water_mask = polys.get("natural", pd.Series(dtype=str)).eq("water")
    # Nature reserves
    reserve_mask = polys.get("leisure", pd.Series(dtype=str)).eq("nature_reserve")

    return {
        "park_area_km2": _area(park_mask),
        "forest_area_km2": _area(forest_mask),
        "water_area_km2": _area(water_mask),
        "nature_reserve_area_km2": _area(reserve_mask),
    }

## Street lamp count

Counts `highway=street_lamp` nodes. Coverage varies by country (Western Europe tends to be better mapped), but this is still a useful direct measure of lighting infrastructure.

In [ ]:
def extract_street_lamps(lat: float, lon: float, city: str) -> dict:
    """
    Count street_lamp nodes within BUFFER_DIST of (lat, lon).
    """
    cached = _load_cache(city, "lamps")
    if cached is not None:
        return {"street_lamp_count": len(cached)}

    try:
        gdf = ox.features_from_point(
            (lat, lon),
            tags={"highway": "street_lamp"},
            dist=BUFFER_DIST,
        )
        _save_cache(gdf, city, "lamps")
        _sleep()
        return {"street_lamp_count": len(gdf)}
    except Exception as e:
        print(f"  WARNING: street lamp extraction failed for {city}: {e}")
        return {"street_lamp_count": np.nan}

## Main extraction loop

Iterates over all 30 cities, calling each extraction function. Cached results are loaded when available to avoid re-querying the Overpass API.

**Expected runtime:** Cities with cached data load in seconds. Uncached cities: ~15-30 min each (roads + 10 km building geometry are the main costs). All results are cached to `data/interim/osm/` — re-runs complete in seconds.

In [ ]:
results = []

for i, row in cities.iterrows():
    city = row["city"]
    lat, lon = row["lat"], row["lon"]
    print(f"[{i + 1}/{len(cities)}] {city} ({lat}, {lon})")

    record = {"city": city}

    # 1. Roads
    road_data = extract_roads(lat, lon, city)
    record.update(road_data)
    print(f"  Roads: {road_data['road_length_total_km']} km total, {road_data['motorway_length_km']} km motorway")

    # 2. Buildings (two-tier: count@25km + area@10km, or full cache if available)
    bldg_data = extract_buildings(lat, lon, city)
    record.update(bldg_data)
    print(f"  Buildings: {bldg_data['building_count']} count, {bldg_data['building_area_km2']} km² area")

    # 3. Land use
    lu_data = extract_landuse(lat, lon, city)
    record.update(lu_data)
    print(f"  Land use: res={lu_data['landuse_residential_km2']}, com={lu_data['landuse_commercial_km2']}, "
          f"ind={lu_data['landuse_industrial_km2']}, ret={lu_data['landuse_retail_km2']}")

    # 4. Green/dark areas
    green_data = extract_green_areas(lat, lon, city)
    record.update(green_data)
    print(f"  Green: park={green_data['park_area_km2']}, forest={green_data['forest_area_km2']}, "
          f"water={green_data['water_area_km2']}, reserve={green_data['nature_reserve_area_km2']}")

    # 5. Street lamps
    lamp_data = extract_street_lamps(lat, lon, city)
    record.update(lamp_data)
    print(f"  Street lamps: {lamp_data['street_lamp_count']}")

    results.append(record)
    print()

print(f"Extraction complete for {len(results)} cities.")

## Assemble and compute derived features

In [ ]:
df_osm = pd.DataFrame(results)

# Compute buffer area for each city (geodesic 25km circle projected to EPSG:3035)
buffer_areas = []
for _, row in cities.iterrows():
    pt = gpd.GeoSeries([Point(row["lon"], row["lat"])], crs=CRS_WGS84)
    pt_proj = pt.to_crs(CRS_PROJ)
    buf = pt_proj.buffer(BUFFER_DIST)
    buffer_areas.append(round(buf.area.iloc[0] / 1e6, 2))

df_osm["buffer_area_km2"] = buffer_areas

# Road density
df_osm["road_density_km_per_km2"] = round(
    df_osm["road_length_total_km"] / df_osm["buffer_area_km2"], 4
)

# Green area total
df_osm["green_area_total_km2"] = round(
    df_osm["park_area_km2"] + df_osm["forest_area_km2"] + df_osm["nature_reserve_area_km2"],
    4,
)

# Built-up fraction
built_up = (
    df_osm["building_area_km2"]
    + df_osm["landuse_commercial_km2"]
    + df_osm["landuse_industrial_km2"]
    + df_osm["landuse_retail_km2"]
)
df_osm["built_up_fraction"] = round(built_up / df_osm["buffer_area_km2"], 4)

# Green fraction
df_osm["green_fraction"] = round(
    df_osm["green_area_total_km2"] / df_osm["buffer_area_km2"], 4
)

# Reorder columns
col_order = [
    "city",
    "road_length_total_km", "motorway_length_km", "road_density_km_per_km2",
    "street_lamp_count",
    "building_count", "building_area_km2",
    "landuse_residential_km2", "landuse_commercial_km2",
    "landuse_industrial_km2", "landuse_retail_km2",
    "park_area_km2", "forest_area_km2", "water_area_km2", "nature_reserve_area_km2",
    "green_area_total_km2", "built_up_fraction", "green_fraction",
    "buffer_area_km2",
]
df_osm = df_osm[col_order]

print(f"Shape: {df_osm.shape}")
df_osm.head(10)

## Validation

In [ ]:
print(f"Expected 30 rows, got {len(df_osm)}")
assert len(df_osm) == 30, f"Row count mismatch: expected 30, got {len(df_osm)}"

# Check merge key matches cities.csv
cities_set = set(cities["city"])
osm_set = set(df_osm["city"])
if cities_set == osm_set:
    print("OK: City names match cities.csv exactly.")
else:
    print(f"MISMATCH: In cities.csv but not OSM: {cities_set - osm_set}")
    print(f"MISMATCH: In OSM but not cities.csv: {osm_set - cities_set}")

# Missing values
print("\nMissing values per column:")
missing = df_osm.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

# Plausibility checks
print("\nPlausibility checks:")
print(f"  road_length_total_km > 0 for all: {(df_osm['road_length_total_km'] > 0).all()}")
print(f"  building_count > 0 for all: {(df_osm['building_count'] > 0).all()}")
print(f"  building_area_km2 > 0 for all (where not NaN): {(df_osm['building_area_km2'].dropna() > 0).all()}")
print(f"  green_area_total_km2 <= buffer_area_km2: {(df_osm['green_area_total_km2'] <= df_osm['buffer_area_km2']).all()}")

print("\nSummary statistics:")
df_osm.describe().T[["mean", "min", "max"]].round(2)

## Quick visual inspection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Road density
df_sorted = df_osm.sort_values("road_density_km_per_km2", ascending=True)
axes[0, 0].barh(df_sorted["city"], df_sorted["road_density_km_per_km2"])
axes[0, 0].set_title("Road density (km/km²)")

# Building area
df_sorted = df_osm.sort_values("building_area_km2", ascending=True)
axes[0, 1].barh(df_sorted["city"], df_sorted["building_area_km2"])
axes[0, 1].set_title("Building footprint area (km²)")

# Green fraction
df_sorted = df_osm.sort_values("green_fraction", ascending=True)
axes[1, 0].barh(df_sorted["city"], df_sorted["green_fraction"])
axes[1, 0].set_title("Green fraction")

# Street lamp count
df_sorted = df_osm.sort_values("street_lamp_count", ascending=True)
axes[1, 1].barh(df_sorted["city"], df_sorted["street_lamp_count"])
axes[1, 1].set_title("Street lamp count")

plt.tight_layout()
plt.show()

## Save to CSV

In [ ]:
output_path = "../data/processed/osm_city_features.csv"
df_osm.to_csv(output_path, index=False)
print(f"Saved {len(df_osm)} rows x {df_osm.shape[1]} columns to {output_path}")

## Limitations

- **OSM is a current snapshot**, not historical data. These features are treated as time-invariant urban characteristics for the study period (2013-2024). In reality, road networks and buildings have changed, but OSM does not provide versioned snapshots through its standard API.
- **Coverage varies by country.** Western European cities (Paris, Amsterdam, Berlin) tend to have much denser OSM mapping than Eastern European cities (Sofia, Bucharest). This may introduce systematic bias in building counts and land use areas.
- **Street lamp coverage** is particularly variable. Some countries/cities have near-complete mapping of street lighting; others have almost none. Interpret this variable with caution.
- **Land use polygons may have gaps.** Not all land within the buffer is necessarily tagged with a land use category in OSM.